# Google Geocoding CSV ツール (Colab)\n\nこのノートブックは `geocode_csv.py` を Google Colab 上で実行するためのものです。

## 1) API キーを設定

In [ ]:
import os
import getpass

os.environ['GOOGLE_MAPS_API_KEY'] = getpass.getpass('Google Maps API Key: ')
print('API キーを環境変数 GOOGLE_MAPS_API_KEY に設定しました。')

## 2) 入力CSVをアップロード

In [ ]:
from google.colab import files

uploaded = files.upload()
print('アップロード済みファイル:', list(uploaded.keys()))

## 3) `geocode_csv.py` を利用する準備（GitHub同期対応）
- **推奨**: GitHub リポジトリを同期して、`geocode_csv.py` を毎回同じバージョンで実行します。
- `GITHUB_REPO_URL` を設定すると、Colab上の作業ディレクトリを毎回作り直して `git clone` します（= ローカル差分や古いファイルをリセット）。
- 未設定の場合は、従来通り `geocode_csv.py` を手動アップロードして使えます。


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 例: 'https://github.com/<owner>/<repo>.git'
# 空文字のままなら、手動アップロードした geocode_csv.py を利用します。
GITHUB_REPO_URL = ''
GITHUB_BRANCH = 'main'
SYNC_DIR = Path('/content/geocode_repo')

if GITHUB_REPO_URL.strip():
    if SYNC_DIR.exists():
        shutil.rmtree(SYNC_DIR)

    clone_cmd = [
        'git',
        'clone',
        '--depth', '1',
        '--branch', GITHUB_BRANCH,
        GITHUB_REPO_URL,
        str(SYNC_DIR),
    ]
    print('同期コマンド:', ' '.join(clone_cmd))
    subprocess.run(clone_cmd, check=True)

    SCRIPT_PATH = SYNC_DIR / 'geocode_csv.py'
    if not SCRIPT_PATH.exists():
        raise FileNotFoundError(
            f'同期先に geocode_csv.py が見つかりません: {SCRIPT_PATH}\n'
            'リポジトリURL・ブランチ・配置場所を確認してください。'
        )
    print(f'同期完了: {SYNC_DIR}')
else:
    SCRIPT_PATH = Path('geocode_csv.py')
    if not SCRIPT_PATH.exists():
        raise FileNotFoundError(
            'geocode_csv.py が見つかりません。CSVと一緒に geocode_csv.py をアップロードするか、'
            'GITHUB_REPO_URL を設定してください。'
        )

print(f'実行スクリプト: {SCRIPT_PATH.resolve()}')



## 4) 変換を実行（ログを時刻付きファイルに保存）
ログは `logs/` 配下に `geocode_YYYYMMDD_HHMMSS.log` 形式で保存します（上書きしません）。


In [ ]:
# 必要に応じてファイル名を変更してください
INPUT_CSV = 'サンプル_飲食店_豊中_緯度経度なし.csv'
OUTPUT_CSV = 'output_with_latlng.csv'

# 任意オプション
DISABLE_PLACES = False
PLACES_BIAS_RADIUS = 3000
SLEEP_SECONDS = 0.1
TIMEOUT_SECONDS = 10


In [ ]:
import os
import subprocess
from datetime import datetime
from pathlib import Path

API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', '').strip()
if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY が未設定です。最初のセルを実行してください。')

def _script_supports_option(script_path: Path, option: str) -> bool:
    """geocode_csv.py --help の出力からオプション有無を判定する。"""
    probe = subprocess.run(
        ['python', str(script_path), '--help'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    return option in probe.stdout

logs_dir = Path('logs')
logs_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = logs_dir / f'geocode_{timestamp}.log'

cmd = [
    'python',
    str(SCRIPT_PATH),
    INPUT_CSV,
    OUTPUT_CSV,
]

# geocode_csv.py のバージョン差異に対応（古い版は未対応オプションを受け付けない）。
if _script_supports_option(SCRIPT_PATH, '--places-bias-radius'):
    cmd.extend(['--places-bias-radius', str(PLACES_BIAS_RADIUS)])
else:
    print('注意: geocode_csv.py が --places-bias-radius 未対応のため、この設定はスキップします。')

if _script_supports_option(SCRIPT_PATH, '--sleep'):
    cmd.extend(['--sleep', str(SLEEP_SECONDS)])
if _script_supports_option(SCRIPT_PATH, '--timeout'):
    cmd.extend(['--timeout', str(TIMEOUT_SECONDS)])
if DISABLE_PLACES:
    if _script_supports_option(SCRIPT_PATH, '--disable-places'):
        cmd.append('--disable-places')
    else:
        print('注意: geocode_csv.py が --disable-places 未対応のため、この設定はスキップします。')

print('実行コマンド:', ' '.join(cmd))
print('ログファイル:', log_path)

with log_path.open('w', encoding='utf-8') as logf:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)

return_code = proc.wait()
print(f'終了コード: {return_code}')
if return_code != 0:
    raise RuntimeError(f'geocode_csv.py の実行に失敗しました。ログを確認してください: {log_path}')

print(f'完了。ログ保存先: {log_path}')



## 5) 出力CSVをダウンロード
必要なら `logs/` 配下のログファイルもダウンロードしてください。


In [ ]:
files.download(OUTPUT_CSV)
